In [3]:
import pandas as pd

df = pd.read_csv("superstore.csv", encoding="utf-8")

print("Shape:", df.shape)
print(df.head())

Shape: (51290, 27)
          Category         City        Country Customer.ID     Customer.Name  \
0  Office Supplies  Los Angeles  United States   LS-172304  Lycoris Saunders   
1  Office Supplies  Los Angeles  United States   MV-174854     Mark Van Huff   
2  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
3  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
4  Office Supplies  Los Angeles  United States   AP-109154    Arthur Prichep   

   Discount Market  记录数               Order.Date        Order.ID  ... Sales  \
0       0.0     US    1  2011-01-07 00:00:00.000  CA-2011-130813  ...    19   
1       0.0     US    1  2011-01-21 00:00:00.000  CA-2011-148614  ...    19   
2       0.0     US    1  2011-08-05 00:00:00.000  CA-2011-118962  ...    21   
3       0.0     US    1  2011-08-05 00:00:00.000  CA-2011-118962  ...   111   
4       0.0     US    1  2011-09-29 00:00:00.000  CA-2011-146969  ...     6   

    Segment              

In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Category        51290 non-null  object 
 1   City            51290 non-null  object 
 2   Country         51290 non-null  object 
 3   Customer.ID     51290 non-null  object 
 4   Customer.Name   51290 non-null  object 
 5   Discount        51290 non-null  float64
 6   Market          51290 non-null  object 
 7   记录数             51290 non-null  int64  
 8   Order.Date      51290 non-null  object 
 9   Order.ID        51290 non-null  object 
 10  Order.Priority  51290 non-null  object 
 11  Product.ID      51290 non-null  object 
 12  Product.Name    51290 non-null  object 
 13  Profit          51290 non-null  float64
 14  Quantity        51290 non-null  int64  
 15  Region          51290 non-null  object 
 16  Row.ID          51290 non-null  int64  
 17  Sales           51290 non-null 

In [5]:
print(df.isnull().sum())

Category          0
City              0
Country           0
Customer.ID       0
Customer.Name     0
Discount          0
Market            0
记录数               0
Order.Date        0
Order.ID          0
Order.Priority    0
Product.ID        0
Product.Name      0
Profit            0
Quantity          0
Region            0
Row.ID            0
Sales             0
Segment           0
Ship.Date         0
Ship.Mode         0
Shipping.Cost     0
State             0
Sub.Category      0
Year              0
Market2           0
weeknum           0
dtype: int64


In [6]:
missing_values = df.isnull().sum()

In [7]:
duplicates = df.duplicated().sum()

print("Duplicates:", duplicates)

Duplicates: 0


In [9]:
import os

quality_report = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

# Create the 'reports' directory if it doesn't exist
output_dir = "reports"
os.makedirs(output_dir, exist_ok=True)

quality_report.to_excel(
    os.path.join(output_dir, "data_quality_report.xlsx"),
    index=False
)

In [10]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

In [12]:
df['Order.Date'] = pd.to_datetime(df['Order.Date'])
df['Ship.Date'] = pd.to_datetime(df['Ship.Date'])

In [14]:
df['Year'] = df['Order.Date'].dt.year

In [16]:
df['Month'] = df['Order.Date'].dt.month_name()

In [18]:
df['Quarter'] = df['Order.Date'].dt.quarter

In [19]:
df.to_csv(
    "cleaned_superstore.csv",
    index=False
)

In [22]:
total_sales = df['Sales'].sum()

total_profit = df['Profit'].sum()

total_orders = df['Order.ID'].nunique()

total_customers = df['Customer.ID'].nunique()

In [23]:
kpi = pd.DataFrame({
    "Metric":[
        "Total Sales",
        "Total Profit",
        "Total Orders",
        "Total Customers"
    ],
    "Value":[
        total_sales,
        total_profit,
        total_orders,
        total_customers
    ]
})

kpi.to_excel(
    "reports/kpi_report.xlsx",
    index=False
)

In [24]:
import matplotlib.pyplot as plt

In [26]:
monthly_sales = (
    df.groupby('Month')['Sales']
      .sum()
)

In [28]:
import os

monthly_sales.plot(kind='bar')

plt.title("Monthly Sales Trend")

plt.tight_layout()

os.makedirs("images", exist_ok=True)
plt.savefig(
    "images/monthly_sales.png"
)

plt.close()

In [29]:
category_sales = (
    df.groupby('Category')['Sales']
      .sum()
)

In [30]:
category_sales.plot(kind='bar')

plt.title("Sales by Category")

plt.tight_layout()

plt.savefig(
    "images/category_sales.png"
)

plt.close()

In [31]:
region_profit = (
    df.groupby('Region')['Profit']
      .sum()
)

In [32]:
region_profit.plot(kind='bar')

plt.title("Profit by Region")

plt.tight_layout()

plt.savefig(
    "images/profit_by_region.png"
)

plt.close()

In [33]:
with pd.ExcelWriter(
    "reports/final_report.xlsx"
) as writer:

    quality_report.to_excel(
        writer,
        sheet_name="Data Quality",
        index=False
    )

    kpi.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )